In [1]:
!pip install fasttext transformers mlflow spacy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.15-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp311-cp311-macosx_11_0_arm64.whl.metadata (9.7 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 15.4 MB/s  0:00:00 eta 0:00:01
Using cached catalogue-2.0.10-py3-none-any.whl (17 kB)
Using cached cymem-2.0.13-cp311-cp311-macosx_11_0_arm64.whl (43 kB)
Using cached murmurhash-1.0.15-cp311-cp311-macosx_11_0_arm64.whl (27 kB)
Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl (29 kB)
Using cached spacy_loggers-1.0.5-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!python -m spacy download ru_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 MB 22.3 MB/s  0:00:01m0:00:0100:01
  Using cached pymorphy3-2.0.6-py3-none-any.whl.metadata (2.4 kB)
  Using cached dawg2_python-0.9.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached pymorphy3_dicts_ru-2.4.417150.4580142-py2.py3-none-any.whl.metadata (2.0 kB)
Using cached pymorphy3-2.0.6-py3-none-any.whl (53 kB)
Using cached dawg2_python-0.9.0-py3-none-any.whl (9.3 kB)
Using cached pymorphy3_dicts_ru-2.4.417150.4580142-py2.py3-none-any.whl (8.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ru-core-news-md] [ru-core-news-md]

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_md')


In [3]:
from abc import ABC, abstractmethod
from enum import Enum

import fasttext
import fasttext.util
import mlflow.transformers
import numpy as np
import spacy
import torch

ImportError: dlopen(/opt/anaconda3/envs/nlp/lib/python3.11/site-packages/torch/_C.cpython-311-darwin.so, 0x0002): Symbol not found: __ZTIN5torch8autograd9generated23LinalgLuFactorBackward0E
  Referenced from: <487E9734-C911-3A2B-BD82-BEE4018A61F1> /opt/anaconda3/envs/nlp/lib/python3.11/site-packages/torch/lib/libtorch_python.dylib
  Expected in:     <03BAFFE3-AB2C-33BF-85F5-18166CD775CD> /opt/anaconda3/envs/nlp/lib/libtorch_cpu.dylib

In [ ]:
MLFLOW_TRACKING_URI = "http://70.34.242.179"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [ ]:
class ToxicityType(str, Enum):
    INSULT = "INSULT"
    NORMAL = "NORMAL"
    OBSCENITY = "OBSCENITY"
    THREAT = "THREAT"


class TextTokenizer(ABC):

    @abstractmethod
    def encode(self, texts: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
        pass

    @abstractmethod
    def decode(self, id: int) -> str:
        pass

class FasttextTokenizer(TextTokenizer):

    FASTTEXT_MODEL_FILE_NAME = "cc.ru.300.bin"

    def __init__(self):
        fasttext.util.download_model("ru", if_exists="ignore")
        self.fasttext_model = fasttext.load_model(self.FASTTEXT_MODEL_FILE_NAME)


    def encode(self, texts: list[str], max_len=30):
        embeddings = []
        zero_vector = torch.zeros(300)
        mask = torch.fill_(torch.zeros(max_len), 1)
        for text in texts:
            words = text.split()[:max_len]
            vectors = [
                torch.from_numpy(self.fasttext_model.get_word_vector(word))
                for word in words
            ]

            if len(vectors) < max_len:
                vectors += [zero_vector] * (max_len - len(vectors))

            embeddings.append(torch.vstack(vectors).unsqueeze(0))
        return torch.vstack(embeddings), mask

    def decode(self, id: int) -> str:
        raise NotImplemented()

class BaseToxicityPredictor(ABC):

    __CLASS_LABELS = ["NORMAL", "INSULT", "THREAT", "OBSCENITY"]
    __CLASS_IDS = [0, 1, 2, 3]
    __CLASS_MAPPING = {
        0: ToxicityType.NORMAL,
        1: ToxicityType.INSULT,
        2: ToxicityType.THREAT,
        3: ToxicityType.OBSCENITY
    }

    def predict(self, text: str) -> str:
        label_id = self.predict_proba(text).argmax()
        return self.__CLASS_MAPPING[label_id]

    @abstractmethod
    def predict_proba(self, text: str) -> np.ndarray:
        pass


class BertToxicityPredictor(BaseToxicityPredictor):

    def __init__(self, model_name, model_alias="final"):
        self.model_name = model_name
        self.model_alias = model_alias
        self.model = mlflow.transformers.load_model(
            model_uri=f"models:/{self.model_name}@{self.model_alias}",
            return_type="pipeline",
            map_location=torch.device("cpu")
        )

    def predict_proba(self, text: str) -> np.ndarray:
        preds = self.model([text], top_k=None)[0]

        preds = sorted(preds, key=lambda item: item["label"])

        return np.array([pred["score"] for pred in preds], dtype=np.float32)


class LSTMToxicityPredictor(BaseToxicityPredictor):

    def __init__(self, model_name="improved_LSTM", model_alias="final"):
        self.model_name = model_name
        self.model_alias = model_alias
        self.tokenizer = FasttextTokenizer()
        self.model = mlflow.pytorch.load_model(
            f"models:/{self.model_name}@{self.model_alias}",
            map_location=torch.device("cpu")
        )

    def predict_proba(self, text: str) -> np.ndarray:
        input, _ = self.tokenizer.encode([text], max_len=len(text))
        probas = self.model(input)
        return probas.detach().numpy()[0]


class LogRegToxicityPredictor(BaseToxicityPredictor):

    __ALLOWED_PUNCT = {'!', '?'}

    def __init__(self, model_name="baseline_logreg_bow", model_alias="final"):
        self.model_name = model_name
        self.model_alias = model_alias
        self.nlp = spacy.load("ru_core_news_md")
        self.model = mlflow.sklearn.load_model(
            f"models:/{self.model_name}@{self.model_alias}"
        )

    def __lemmatize(self, text: str):
        cleaned = []
        for token in self.nlp(text):
            if token.is_stop:
                continue
            if token.is_alpha:
                lemma = token.lemma_
                if len(lemma) < 3 or len(lemma) > 30:
                    continue
                else:
                    cleaned.append(lemma)
            elif token.is_punct:
                if token.text in self.__ALLOWED_PUNCT:
                    cleaned.append(token.text)
            else:
                cleaned.append(token.text)
        return ' '.join(cleaned)

    def predict_proba(self, text: str) -> np.ndarray:
        lemmatized = self.__lemmatize(text)
        return self.model.predict_proba([lemmatized])[0]



In [ ]:
big_bert_predictor = BertToxicityPredictor("BERT_clf_dropout_DeepPavlov_rubert-base-cased-conversational")
tiny_bert_predictor = BertToxicityPredictor("BERT_clf_dropout_cointegrated_rubert-tiny2")
lstm_predictor = LSTMToxicityPredictor()
logreg_predictor = LogRegToxicityPredictor()

In [ ]:
text = "нормальный текст"

print(big_bert_predictor.predict(text))
print(tiny_bert_predictor.predict(text))
print(lstm_predictor.predict(text))
print(logreg_predictor.predict(text))